# 7. Cluster claims

For each paper, cluster the per-post endorsement `claim_text`s into 3..7 canonical claims via one Haiku call per paper.

Inputs: `output/<RUN_ID>/classified.parquet` from notebook 5.
Outputs: `output/<RUN_ID>/clusters.json` shaped as `{ ro_id: [{canonical_text, member_post_ids}, ...] }`.

See `PIPELINE_PLAN.md` § S24.

## Setup

Pick the latest `RUN_ID` that has a `classified.parquet`, load `.env`, and build the Anthropic client.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent.resolve()
OUTPUT_ROOT = REPO_ROOT / "altendor" / "output"

load_dotenv(REPO_ROOT / ".env")

# Override RUN_ID here if you want a specific historical run instead of the latest.
RUN_ID: str | None = None
if RUN_ID is None:
    candidates = sorted([p.name for p in OUTPUT_ROOT.iterdir() if (p / "classified.parquet").exists()])
    if not candidates:
        raise FileNotFoundError("No run with classified.parquet found. Run notebook 5 first.")
    RUN_ID = candidates[-1]

OUT_DIR = OUTPUT_ROOT / RUN_ID
print(f"Using run id: {RUN_ID}\nOutput dir: {OUT_DIR}")

import anthropic

client = anthropic.Anthropic()

In [ ]:
import pandas as pd

classified_df = pd.read_parquet(OUT_DIR / "classified.parquet")
endorsements = classified_df[classified_df["kind"] == "endorsement"].copy()
print(f"{len(endorsements)} endorsements across {endorsements['ro_id'].nunique()} papers")

## Per-paper clustering

One Haiku call per paper. The function aims for 3..7 canonical claims; on failure it falls back gracefully but we still wrap in `try/except` for belt-and-braces.

In [ ]:
from dataclasses import asdict

from altendor.cluster.claims import cluster_claims

clusters_by_paper: dict[str, list[dict]] = {}

for ro_id, group in endorsements.groupby("ro_id"):
    claim_texts = dict(zip(group["post_id"].astype(str), group["claim_text"].fillna("").astype(str)))
    claim_texts = {pid: txt for pid, txt in claim_texts.items() if txt.strip()}
    if not claim_texts:
        continue
    try:
        clusters = cluster_claims(client, claim_texts)
    except Exception as exc:  # the function already falls back, but belt + braces
        print(f"  {ro_id}: clustering failed ({exc}); skipping")
        continue
    clusters_by_paper[ro_id] = [asdict(c) for c in clusters]
    print(f"  {ro_id}: {len(claim_texts)} claims → {len(clusters)} clusters")

print(f"\nClustered {len(clusters_by_paper)} papers.")

## Quick summary

Total clusters produced and a histogram of cluster sizes (members per canonical claim).

In [ ]:
n_clusters_total = sum(len(v) for v in clusters_by_paper.values())
print(f"Total clusters across all papers: {n_clusters_total}")

cluster_size_hist = pd.Series(
    [len(c["member_post_ids"]) for v in clusters_by_paper.values() for c in v]
).value_counts().sort_index()
print("\nMembers-per-cluster histogram:")
print(cluster_size_hist.to_string())

## Write outputs

Persist `clusters.json` and append to the run's `manifest.json`.

In [ ]:
import json

out_path = OUT_DIR / "clusters.json"
out_path.write_text(json.dumps(clusters_by_paper, indent=2, sort_keys=True))

manifest_path = OUT_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
manifest["stage_7_cluster_claims"] = {
    "n_papers_clustered": len(clusters_by_paper),
    "n_clusters_total": int(n_clusters_total),
}
manifest.setdefault("output_files", []).append(str(out_path.relative_to(REPO_ROOT)))
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True))
print(f"Wrote {out_path}")